# Apply the saved PM2.5 calibration models

Apply the existing low- and high-concentration H2O models without training a new calibration model. The models were developed in Qian et al. (2026), Atmospheric Measurement Techniques 19, 603-615; their training code is archived at https://doi.org/10.5281/zenodo.18305879.

**Step:** 02 (optional, upstream). Apply the saved PM2.5 calibration models.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Long hourly measurements plus `modCalib_lowlvl` and `modCalib_highlvl`; a matching H2O release and Java are required for scoring.

**Outputs:** A calibrated long hourly CSV. Scoring is disabled by default; see README.md for the separate calibration environment.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure calibration explicitly

The manuscript already has a saved calibrated input. Set `RUN_CALIBRATION = True` only when intentionally reproducing this upstream step in the compatible calibration environment. The threshold chooses the low model for raw PM2.5 below 50 and the high model otherwise. It is not the event-eligibility threshold of 100.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

from types import SimpleNamespace
from shared import data_file, repository_path
import tempfile
RUN_CALIBRATION = os.environ.get("INDOOR_AQ_RUN_CALIBRATION", "0") == "1"
calibration = SimpleNamespace(
    input=os.environ.get("INDOOR_AQ_RAW_INPUT") or str(output_root(OUTPUT) / "data/indoor_all_long_hourly.csv"),
    output=str(output_root(OUTPUT) / "data/indoor_all_long_hourly_calibrated.csv"),
    models_dir=str(repository_path(os.environ.get("INDOOR_AQ_CALIBRATION_MODELS", DATA / "models"))),
    h2o_pydeps="",  # Use the active environment unless an explicit dependency directory is required.
    h2o_temp=str(Path(tempfile.gettempdir()) / "indoor_aq_h2o"),
    threshold=50.0,
)

## Import numeric and filesystem helpers

H2O is imported only inside the scoring path. The configuration and function definitions can be inspected without starting a Java server.

In [ ]:
import os
import sys
from pathlib import Path
import tempfile
from shared import DATA, output_root

import numpy as np
import pandas as pd

## Load h2o

Initialize the requested H2O environment and a 4 GB Java server. The saved binary models require a compatible H2O release. The temporary directory is separate from manuscript data.

In [ ]:
def load_h2o(h2o_pydeps: Path, h2o_temp: Path):
    if str(h2o_pydeps) not in ('', '.'):
        sys.path.insert(0, str(h2o_pydeps))
    import h2o  # noqa: PLC0415

    h2o_temp.mkdir(parents=True, exist_ok=True)
    os.environ["TEMP"] = str(h2o_temp)
    os.environ["TMP"] = str(h2o_temp)
    h2o.init(max_mem_size="4G", nthreads=-1, ice_root=str(h2o_temp))
    return h2o

## Score subset

Score the model's exact predictor columns: temperature (`atmp`), RH (`rhum`), and raw PM2.5 (`pm02_refs`). Empty subsets return an empty numeric array. Remove temporary H2O frames after scoring.

In [ ]:
def score_subset(h2o, model, score_df: pd.DataFrame) -> np.ndarray:
    if score_df.empty:
        return np.array([], dtype=float)

    h2o_frame = h2o.H2OFrame(score_df[["atmp", "rhum", "pm02_refs"]])
    pred = model.predict(h2o_frame).as_data_frame(use_pandas=True)["predict"]
    h2o.remove(h2o_frame)
    return pred.to_numpy(dtype=float)

## Format prediction

Write finite predictions with up to ten decimal places and leave nonfinite values blank. This avoids converting a missing prediction into a measured zero.

In [ ]:
def format_prediction(value: float) -> str:
    if not np.isfinite(value):
        return ""
    return f"{value:.10f}".rstrip("0").rstrip(".")

## Define the saved-model application

Preserve original row order and all non-PM2.5 values. Build a sensor-time predictor table, split on the raw concentration threshold, score the two saved models, and merge only PM2.5 predictions back. The H2O server is shut down in `finally`, including when scoring fails. No fitted model is modified.

In [ ]:
def apply_calibration(args) -> None:
    input_path = Path(args.input)
    output_path = Path(args.output)
    models_dir = Path(args.models_dir)
    h2o_pydeps = Path(args.h2o_pydeps)
    h2o_temp = Path(args.h2o_temp)

    if not input_path.exists():
        raise FileNotFoundError(input_path)
    if not (models_dir / "modCalib_lowlvl").exists():
        raise FileNotFoundError(models_dir / "modCalib_lowlvl")
    if not (models_dir / "modCalib_highlvl").exists():
        raise FileNotFoundError(models_dir / "modCalib_highlvl")

    print(f"Reading raw data: {input_path}")
    raw_text = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    expected_cols = ["sensor_id", "variable", "time", "value"]
    if list(raw_text.columns) != expected_cols:
        raise ValueError(f"Expected columns {expected_cols}, found {list(raw_text.columns)}")

    raw_text["_row_order"] = np.arange(len(raw_text), dtype=np.int64)
    raw_numeric = raw_text.copy()
    raw_numeric["value"] = pd.to_numeric(raw_numeric["value"], errors="coerce")

    print("Building PM/RH/temperature scoring table...")
    wide = (
        raw_numeric.pivot_table(
            index=["sensor_id", "time"],
            columns="variable",
            values="value",
            aggfunc="first",
            observed=False,
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )

    required_vars = {"pm25", "rh", "temp"}
    missing_vars = required_vars.difference(wide.columns)
    if missing_vars:
        raise ValueError(f"Missing required variable(s) in input data: {sorted(missing_vars)}")

    score = wide.loc[wide["pm25"].notna(), ["sensor_id", "time", "pm25", "rh", "temp"]].copy()
    score = score.rename(columns={"pm25": "pm02_refs", "rh": "rhum", "temp": "atmp"})
    score["calibrated_pm25"] = np.nan

    low_mask = score["pm02_refs"] < args.threshold
    high_mask = ~low_mask
    print(f"Rows to score: {len(score):,} ({low_mask.sum():,} low, {high_mask.sum():,} high)")

    h2o = load_h2o(h2o_pydeps, h2o_temp)
    try:
        model_low = h2o.load_model(str(models_dir / "modCalib_lowlvl"))
        model_high = h2o.load_model(str(models_dir / "modCalib_highlvl"))

        score.loc[low_mask, "calibrated_pm25"] = score_subset(h2o, model_low, score.loc[low_mask])
        score.loc[high_mask, "calibrated_pm25"] = score_subset(h2o, model_high, score.loc[high_mask])
    finally:
        h2o.cluster().shutdown(prompt=False)

    predictions = score[["sensor_id", "time", "calibrated_pm25"]].copy()
    predictions["calibrated_pm25"] = predictions["calibrated_pm25"].map(format_prediction)
    calibrated = raw_text.merge(predictions, on=["sensor_id", "time"], how="left", sort=False)
    pm25_mask = calibrated["variable"].eq("pm25") & calibrated["calibrated_pm25"].notna()
    calibrated.loc[pm25_mask, "value"] = calibrated.loc[pm25_mask, "calibrated_pm25"]

    calibrated = calibrated.sort_values("_row_order", kind="stable")[expected_cols]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    calibrated.to_csv(output_path, index=False)

    raw_pm = pd.to_numeric(raw_text.loc[raw_text["variable"].eq("pm25"), "value"], errors="coerce")
    cal_pm = pd.to_numeric(calibrated.loc[calibrated["variable"].eq("pm25"), "value"], errors="coerce")
    print(f"Saved calibrated data: {output_path}")
    print(f"Rows written: {len(calibrated):,}")
    print(f"PM2.5 non-missing raw/calibrated: {raw_pm.notna().sum():,}/{cal_pm.notna().sum():,}")
    print(
        "PM2.5 mean raw/calibrated: "
        f"{raw_pm.mean(skipna=True):.3f}/{cal_pm.mean(skipna=True):.3f}"
    )

## Apply calibration when requested

Scoring is disabled by default. Enable `RUN_CALIBRATION` to read the output of step 01 (or an explicitly supplied long hourly file), apply the two saved models, and write the calibrated input for step 03. If scoring is skipped, supply calibrated measurements for step 03.

In [ ]:
if RUN_CALIBRATION:
    calibration.input = str(repository_path(os.environ['INDOOR_AQ_RAW_INPUT'])) if os.environ.get('INDOOR_AQ_RAW_INPUT') else str(data_file('indoor_all_long_hourly.csv', OUTPUT))
    apply_calibration(calibration)
else:
    print("Calibration skipped (RUN_CALIBRATION=False). Step 03 requires supplied calibrated measurements.")